# NLP: Reasoning with Verification

## Task Description
Implement a 'System 2' reasoning loop where the model critiques and corrects its own output.

## License

This notebook is licensed under the Apache License 2.0. See the [LICENSE](https://github.com/huggingface/transformers/blob/main/LICENSE) file for more details.

This notebook focuses on open-source, lightweight, and updated models, and is optimized to run on the Google Colab Free Tier.

## Learning Objectives

*   **Practical:** Gain hands-on experience with this model.
*   **Theory:** Understand the underlying concepts and architecture.

In [ ]:
# Check for GPU
!nvidia-smi

## 1. Installation
Install necessary libraries.

In [ ]:
!pip install -q transformers torch accelerate

## 2. Imports
Import libraries.

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

## 3. Load Data
Load dataset or input data.

In [ ]:
problem = "I have 3 apples. I eat 2. Then I buy 5 more. How many do I have?" 

## 4. Data Cleaning
Preprocess the data.

In [ ]:
# System prompt for verification loop
system_prompt = "You are a careful reasoner. Solve the problem, then verify your answer step by step." 

## 5. Load Model
Load the model.

In [ ]:
model_id = "Qwen/Qwen2.5-1.5B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(model_id, torch_dtype=torch.float16, device_map="cuda")

## 6. Inference
Run the model.

In [ ]:
# 1. Initial Solution
messages = [
    {"role": "system", "content": system_prompt},
    {"role": "user", "content": problem}
]
text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
inputs = tokenizer(text, return_tensors="pt").to(model.device)
outputs = model.generate(**inputs, max_new_tokens=200)
initial_response = tokenizer.decode(outputs[0], skip_special_tokens=True)

# 2. Verification Step (Self-Correction)
verify_msg = messages + [
    {"role": "assistant", "content": initial_response},
    {"role": "user", "content": "Double check your reasoning. Are there any mistakes? Output the final correct answer."}
]
text_v = tokenizer.apply_chat_template(verify_msg, tokenize=False, add_generation_prompt=True)
inputs_v = tokenizer(text_v, return_tensors="pt").to(model.device)
outputs_v = model.generate(**inputs_v, max_new_tokens=200)
final_response = tokenizer.decode(outputs_v[0], skip_special_tokens=True)

print("--- Initial ---")
print(initial_response)
print("\n--- Verified ---")
print(final_response)
results = final_response

## 7. Save Results
Save outputs.

In [ ]:
# Kitchen Sink: Save Results
import os
import numpy as np
try:
    from PIL import Image
except ImportError:
    pass
try:
    import scipy.io.wavfile as wavfile
except ImportError:
    pass

print("Saving results...")

# 1. Text / Data / Variables
with open('results.txt', 'w') as f:
    if 'results' in locals():
        f.write(str(results))
    elif 'predictions' in locals():
        f.write(str(predictions))
    elif 'prediction' in locals():
        f.write(str(prediction))
    elif 'output' in locals():
        f.write(str(output))
    elif 'features' in locals():
        f.write(str(features))
    elif 'summary' in locals():
        f.write(str(summary))
    elif 'translation' in locals():
        f.write(str(translation))
    elif 'answers' in locals():
        f.write(str(answers))
    elif 'conversation' in locals():
        f.write(str(conversation))
    elif 'generated_text' in locals():
        f.write(str(generated_text))
    elif 'transcription' in locals():
        f.write(str(transcription))
    elif 'text' in locals():
        f.write(str(text))
    elif 'triplets' in locals():
        f.write(str(triplets))
    else:
        f.write("Task completed.")
print("Saved results.txt")

# 2. Images
if 'image' in locals():
    if hasattr(image, 'save'):
        image.save('generated_image.png')
        print("Saved generated_image.png")
elif 'images' in locals() and isinstance(images, list) and len(images) > 0:
    if hasattr(images[0], 'save'):
        images[0].save('generated_image.png')
        print("Saved generated_image.png (first image)")
    # Try to save as GIF if it's a list of images (for video/3D)
    try:
        from diffusers.utils import export_to_gif
        export_to_gif(images, "generated_video.gif")
        print("Saved generated_video.gif")
    except:
        # Fallback for standard list of PIL images
        try:
            images[0].save("generated_video.gif", save_all=True, append_images=images[1:], duration=100, loop=0)
            print("Saved generated_video.gif")
        except:
            pass

# Graph Visualization (specific to Graph RAG)
if os.path.exists("graph_visualization.png"):
    import shutil
    shutil.copy("graph_visualization.png", "generated_image.png")
    print("Saved generated_image.png from graph_visualization.png")

# 3. Audio
if 'audio' in locals():
    if isinstance(audio, tuple):
        rate, data = audio
        try:
            wavfile.write('generated_audio.wav', rate, data)
            print("Saved generated_audio.wav")
        except:
            pass
    elif hasattr(audio, 'ndim'): # numpy array
        rate = save_rate if 'save_rate' in locals() else 16000
        try:
            wavfile.write('generated_audio.wav', rate, audio)
            print("Saved generated_audio.wav")
        except:
            pass
elif 'audio_data' in locals():
    rate = save_rate if 'save_rate' in locals() else 16000
    try:
        wavfile.write('generated_audio.wav', rate, audio_data)
        print("Saved generated_audio.wav")
    except:
        pass


## 8. Advanced Usage
Explore further.

**Teaching Tip:** Try varying the input parameters or prompts to see how the model's output changes. This helps build intuition about the model's capabilities and limitations.

In [ ]:
print('Try different inputs!')